In [1]:
# Preprocess CESM2 PM2.5 data
# Before adding a new scenario, check that the data matches

In [2]:
import os
import cftime
import xarray as xr
import warnings
from utils.utils import get_scenario_config, load_file_list

In [3]:
# This will depend on the model, for CESM2 the surface is the final
# variable in the "lev" coord
def select_surface(da):
    return da.isel(lev=-1)


# Conversion functions
def kgm3_to_µgm3(data):
    # kg/m3 -> µg/m3 (multiply by 1e9)
    return data * 1e9


# Load file, select surface and convert units
def load_pm25_µgm3(ds, scenario):
    config = SCENARIO_CONFIG[scenario]
    var_name = config["var"]
    converter = config["convert"]

    data = ds[var_name]
    surface = select_surface(data)
    converted = converter(surface)
    converted.attrs["units"] = "µg/m3"
    return converted


# CESM2 often has months shifted one i.e.
# the data represents 2015-01 - 2020-12 but time coord shows 2015-02 - 2021-01
def minus_one_month(date):
    """Subtract one month from a cftime.DatetimeNoLeap object."""
    year, month = date.year, date.month
    if month == 1:
        return cftime.DatetimeNoLeap(year - 1, 12, date.day,
                                     date.hour, date.minute, date.second,
                                     date.microsecond, has_year_zero=date.has_year_zero)
    else:
        return cftime.DatetimeNoLeap(year, month - 1, date.day,
                                     date.hour, date.minute, date.second,
                                     date.microsecond, has_year_zero=date.has_year_zero)

In [4]:
# Map scenario -> variable name + conversion
# This may be different across scenarios for other models - CHECK
SCENARIO_CONFIG = {
    "SSP245_G6": {
        "var": "PM25",
        "convert": kgm3_to_µgm3},
    "G6-1.5K": {
        "var": "PM25",
        "convert": kgm3_to_µgm3},
    "hist": {
        "var": "PM25",
        "convert": kgm3_to_µgm3},
}

In [9]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "hist"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/file_paths/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/monthly_pm25/"

In [10]:
# === MAIN LOOP ===

for ens_num in ensemble_members:
    print(f"Processing {scenario}, ensemble {ens_num:02d}")
    file_list = load_file_list(FILE_DIR, f"file_list_{scenario}_{ens_num}.json")
    datasets = []

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        da = load_pm25_µgm3(xr.open_dataset(f), scenario)
        datasets.append(da)

    # Combine files if multiple
    combined_da = xr.concat(datasets,
                            dim="time") if len(datasets) > 1 else datasets[0]

    # If the first month is February (2) then apply month fixer
    first_month = combined_da.time.dt.month[0]
    if first_month == 2:
        print("Adjusting month indexing")
        new_time = [minus_one_month(t) for t in combined_da["time"].values]
        combined_da = combined_da.assign_coords(time=new_time)
    # If the first month is January (1) don't apply month fixer
    elif first_month == 1:
        print("No month index adjusting needed")
    else:
        warnings.warn(f"First month: {first_month}, check dates in file")

    # Slice time to match years for each scenario
    sliced_da = combined_da.sel(time=slice(str(years.start), str(years.stop)))
    # Remove the unused lev dimension
    sliced_da = sliced_da.drop_vars("lev")

    dates = f"{years.start}-{years.stop}"

    description = ("Processed monthly surface PM2.5 "
                   "- scripts by A.F. Wells (2025)")
    sliced_da.attrs["description"] = description
    sliced_da.attrs["ensemble_number"] = ens_num
    sliced_da.attrs["scenario"] = scenario
    sliced_da.attrs["model"] = model

    out_file = f"Monthly_PM25_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    sliced_da.to_netcdf(out_path)

print("All processing complete.")

Processing hist, ensemble 01
Reading b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h0.PM25.197801-199912.nc
Reading b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h0.PM25.199912-201412.nc
Adjusting month indexing
Saving to /glade/work/awells/air_quality/CESM2/pm25/monthly_pm25/Monthly_PM25_CESM2_hist_01_1990-2010.nc
All processing complete.
